<a href="https://colab.research.google.com/github/johnmichaels4867-create/ACCTG5150/blob/main/labs_hw/week6_ETL-SQL-LLM/Lab-6_SQL-with-DuckDB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 6 - SQL with DuckDB

<a href="https://colab.research.google.com/github/gaulinmp/AccountingDataAnalytics/blob/main/labs_hw/week6_ETL-SQL-LLM/Lab-6_SQL-with-DuckDB.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

This notebook will walk you through loading two datasets and joining them together with SQL. We will cover the following steps:

1. Loading the Compustat (annual financial statements) and CRSP (daily stock prices) data
2. Querying the data with SQL, using a library called DuckDB

The intent of the notebook is for you to execute each cell in order. Before you start, drag both `lab6_data_compustat.csv` and `lab6_data_crsp.csv` into your Colab page (see Lab 1 for instructions).

In [1]:
# The usual imports: pathlib to point to files, and pandas to load them
from pathlib import Path
import pandas as pd
# This one is new. DuckDB is a small, fast database that lives inside Python.
# It lets us write SQL queries directly against pandas DataFrames.
import duckdb

In [5]:
# Point to the files, and make sure they exist
compustat_path = Path("lab6_data_compustat.csv")
crsp_path = Path("lab6_data_crsp.csv")
compustat_path.exists(), crsp_path.exists()

(True, True)

That should say `(True, True)` above. If either says `False`, then that file is not loaded correctly. Please make sure to drag both CSV files into your Colab page.

Okay, now to actually load the data. Notice the `parse_dates` argument: it tells pandas which columns are dates, so they get loaded as dates instead of text. That matters later, because we're going to compare dates in our SQL query.

In [6]:
# Load Compustat: one row per company per fiscal year
compustat = pd.read_csv(compustat_path, parse_dates=["fiscal_year_end", "fiscal_year_end_prev", "fiscal_year_end_next",
                                                     "earn_annc_date", "earn_annc_date_prev", "earn_annc_date_next"])
compustat.head()

,name,fyear,ticker,permno,fyr,earn_annc_date,age_days,fiscal_year_end,at,act,...,bve,bign,auditor,auop,fiscal_year_end_prev,earn_annc_date_prev,fiscal_year_end_next,earn_annc_date_next,gics_sector,gics_sector_name
0,AMERICAN AIRLINES GROUP INC,2000,AMR,21020,12,2001-01-17,18263,2000-12-31,26213.0,5179.0,...,7176.0,1,Ernst & Young,Unqualified,1999-12-31,2000-01-19,2001-12-31,2002-01-16,20.0,Industrials
1,AMERICAN AIRLINES GROUP INC,2001,AMR,21020,12,2002-01-16,18628,2001-12-31,32841.0,6540.0,...,5373.0,1,Ernst & Young,Unqualified with additional language,2000-12-31,2001-01-17,2002-12-31,2003-01-22,20.0,Industrials
2,AMERICAN AIRLINES GROUP INC,2002,AMR,21020,12,2003-01-22,18993,2002-12-31,30267.0,4937.0,...,957.0,1,Ernst & Young,Unqualified with additional language,2001-12-31,2002-01-16,2003-12-31,2004-01-21,20.0,Industrials
3,AMERICAN AIRLINES GROUP INC,2003,AMR,21020,12,2004-01-21,19358,2003-12-31,29330.0,4682.0,...,46.0,1,Ernst & Young,Unqualified with additional language,2002-12-31,2003-01-22,2004-12-31,2005-01-19,20.0,Industrials
4,AMERICAN AIRLINES GROUP INC,2004,AMR,21020,12,2005-01-19,19724,2004-12-31,28773.0,4971.0,...,-581.0,1,Ernst & Young,Unqualified with additional language,2003-12-31,2004-01-21,2005-12-31,2006-01-18,20.0,Industrials


In [7]:
# Load CRSP: one row per company per trading day
crsp = pd.read_csv(crsp_path, parse_dates=["date"])
crsp.head()

,permno,ticker,date,ret,prc,shrout,vol,bid,ask,shrcls
0,10107,MSFT,1999-01-04,0.016674,141.000,2470000,17868249,140.875,140.9375,A
1,10107,MSFT,1999-01-05,0.039007,146.500,2470000,16424412,146.375,146.4375,A
2,10107,MSFT,1999-01-06,0.032423,151.250,2470000,18025141,151.375,151.4375,A
3,10107,MSFT,1999-01-07,-0.004959,150.500,2470000,14179585,150.500,150.5000,A
4,10107,MSFT,1999-01-08,-0.004153,149.875,2470000,12709729,149.875,149.9375,A


Take a second to notice how different these two tables are. Compustat has one row per firm-*year* (a few hundred rows), while CRSP has one row per firm-*day* (over 150,000 rows). Let's confirm:

In [8]:
print("Compustat:", compustat.shape)
print("CRSP:     ", crsp.shape)

Compustat: (594, 49)
CRSP:      (156002, 10)


Both tables have a `permno` column, which is CRSP's permanent identifier for a stock. That's our key for linking the two tables together. (Why not `ticker`? Because tickers change. American Airlines was `AMR` before it was `AAL`, but its `permno` stayed the same.)

## SQL with DuckDB

Now for the fun part. We want to match each firm's *daily* stock data to the *annual* financial statements that were current on that day. In other words, for each fiscal year, grab every trading day from that fiscal year end up to (but not including) the next one.

That's a join on `permno`, plus a date range condition. Here's the SQL:

```sql
-- Example Query: Join Compustat and CRSP on Permno and Date between fiscal year end dates
SELECT *
FROM compustat
JOIN crsp
    ON compustat.permno = crsp.permno
        AND date >= fiscal_year_end
        AND date < fiscal_year_end_next
LIMIT 5
```

The neat trick with DuckDB is that the table names in the query (`compustat` and `crsp`) are just the names of our pandas DataFrames. DuckDB finds them automatically. We write the query as a Python string, hand it to `duckdb.sql()`, and convert the result back to a DataFrame with `.df()`.

In [9]:
query = """
-- Example Query: Join Compustat and CRSP on Permno and Date between fiscal year end dates
SELECT *
FROM compustat
JOIN crsp
    ON compustat.permno = crsp.permno
        AND date >= fiscal_year_end
        AND date < fiscal_year_end_next
LIMIT 5
"""

duckdb.sql(query).df()

,name,fyear,ticker,permno,fyr,earn_annc_date,age_days,fiscal_year_end,at,act,...,permno_1,ticker_1,date,ret,prc,shrout,vol,bid,ask,shrcls
0,MICROSOFT CORP,2006,MSFT,10107,6,2006-07-20,7670,2006-06-30,69597.0,49010.0,...,10107,MSFT,2006-06-30,-0.007252,23.30,10062000,73495446,23.38,23.31,A
1,MICROSOFT CORP,2006,MSFT,10107,6,2006-07-20,7670,2006-06-30,69597.0,49010.0,...,10107,MSFT,2006-07-03,0.017167,23.70,10062000,25744996,23.67,23.68,A
2,MICROSOFT CORP,2006,MSFT,10107,6,2006-07-20,7670,2006-06-30,69597.0,49010.0,...,10107,MSFT,2006-07-05,-0.014768,23.35,10062000,53172363,23.33,23.31,A
3,MICROSOFT CORP,2006,MSFT,10107,6,2006-07-20,7670,2006-06-30,69597.0,49010.0,...,10107,MSFT,2006-07-06,0.005567,23.48,10062000,63802278,23.46,23.47,A
4,MICROSOFT CORP,2006,MSFT,10107,6,2006-07-20,7670,2006-06-30,69597.0,49010.0,...,10107,MSFT,2006-07-07,-0.007666,23.30,10062000,65168568,23.31,23.31,A


Scroll all the way to the right in that output. You'll see columns named `permno_1` and `ticker_1`. That's because both tables have a `permno` and `ticker` column, and `SELECT *` grabs everything from both. DuckDB renames the duplicates so they don't collide.

In practice, you'd usually pick just the columns you want instead of `*`. Here's the same join, returning only the firm name, fiscal year, net income (`ni`), the trading date, and the daily return (`ret`). Notice the `table.column` syntax, which tells SQL which table each column comes from.

In [10]:
query = """
SELECT compustat.name, compustat.fyear, compustat.ni, crsp.date, crsp.ret
FROM compustat
JOIN crsp
    ON compustat.permno = crsp.permno
        AND date >= fiscal_year_end
        AND date < fiscal_year_end_next
LIMIT 5
"""

duckdb.sql(query).df()

,name,fyear,ni,date,ret
0,MICROSOFT CORP,2006,12599.0,2006-06-30,-0.007252
1,MICROSOFT CORP,2006,12599.0,2006-07-03,0.017167
2,MICROSOFT CORP,2006,12599.0,2006-07-05,-0.014768
3,MICROSOFT CORP,2006,12599.0,2006-07-06,0.005567
4,MICROSOFT CORP,2006,12599.0,2006-07-07,-0.007666


Now you're writing SQL. From here, you can add a `WHERE` clause to filter, `GROUP BY` to aggregate (e.g., the average daily return per firm-year), or `ORDER BY` to sort. And remember: if you get stuck, write a comment describing what you want, and let Gemini suggest the query.

In [14]:
query = """
SELECT c.name, c.fyear, c.fiscal_year_end, r.date, r.ret
FROM compustat AS c
LEFT JOIN crsp AS r
    ON c.permno = r.permno
   AND c.fiscal_year_end = r.date
ORDER BY c.name, c.fyear
LIMIT 20
"""

duckdb.sql(query).df()

,name,fyear,fiscal_year_end,date,ret
0,ADVANCED MICRO DEVICES,2000,2000-12-31,NaT,NaN
1,ADVANCED MICRO DEVICES,2001,2001-12-31,2001-12-31,-0.031746
2,ADVANCED MICRO DEVICES,2002,2002-12-31,2002-12-31,0.017323
3,ADVANCED MICRO DEVICES,2003,2003-12-31,2003-12-31,-0.013898
4,ADVANCED MICRO DEVICES,2004,2004-12-31,2004-12-31,0.003189
5,ADVANCED MICRO DEVICES,2005,2005-12-31,NaT,NaN
6,ADVANCED MICRO DEVICES,2006,2006-12-31,NaT,NaN
7,ADVANCED MICRO DEVICES,2007,2007-12-31,2007-12-31,0.024590
8,ADVANCED MICRO DEVICES,2008,2008-12-31,2008-12-31,-0.022624
9,ADVANCED MICRO DEVICES,2009,2009-12-31,2009-12-31,-0.004115


In [15]:
query = """
WITH crsp_n AS (
    SELECT permno, date, ret,
           ROW_NUMBER() OVER (PARTITION BY permno ORDER BY date) AS rn
    FROM crsp
),
ea AS (
    SELECT c.permno, c.name, c.fyear, c.earn_annc_date,
           (SELECT MIN(n.rn)
            FROM crsp_n n
            WHERE n.permno = c.permno
              AND n.date >= c.earn_annc_date) AS rn0
    FROM compustat AS c
)
SELECT e.name, e.fyear, e.earn_annc_date,
       m1.date AS date_m1, m1.ret AS ret_m1,
       m0.date AS date_0,  m0.ret AS ret_0,
       p1.date AS date_p1, p1.ret AS ret_p1
FROM ea AS e
LEFT JOIN crsp_n AS m1 ON m1.permno = e.permno AND m1.rn = e.rn0 - 1
LEFT JOIN crsp_n AS m0 ON m0.permno = e.permno AND m0.rn = e.rn0
LEFT JOIN crsp_n AS p1 ON p1.permno = e.permno AND p1.rn = e.rn0 + 1
ORDER BY e.name, e.fyear
"""

duckdb.sql(query).df()

,name,fyear,earn_annc_date,date_m1,ret_m1,date_0,ret_0,date_p1,ret_p1
0,ADVANCED MICRO DEVICES,2000,2001-01-17,2001-01-16,-0.038597,2001-01-17,0.080292,2001-01-18,0.226351
1,ADVANCED MICRO DEVICES,2001,2002-01-16,2002-01-15,-0.023457,2002-01-16,-0.065274,2002-01-17,-0.034637
2,ADVANCED MICRO DEVICES,2002,2003-01-16,2003-01-15,0.002692,2003-01-16,-0.033557,2003-01-17,-0.162500
3,ADVANCED MICRO DEVICES,2003,2004-01-20,2004-01-16,0.054088,2004-01-20,0.036993,2004-01-21,-0.085155
4,ADVANCED MICRO DEVICES,2004,2005-01-18,2005-01-14,0.000000,2005-01-18,0.022427,2005-01-19,0.000000
...,...,...,...,...,...,...,...,...,...
589,WALMART INC,2021,2022-02-17,2022-02-16,-0.006251,2022-02-17,0.040066,2022-02-18,-0.006408
590,WALMART INC,2022,2023-02-21,2023-02-17,0.015041,2023-02-21,0.006078,2023-02-22,-0.020973
591,WALMART INC,2023,2024-02-20,2024-02-16,0.006321,2024-02-20,0.032285,2024-02-21,-0.012283
592,WALMART INC,2024,2025-02-20,2025-02-19,0.002120,2025-02-20,-0.065289,2025-02-21,-0.024997


In [16]:
query = """
SELECT c.name, c.fyear, c.permno, c.fiscal_year_end,
       c.earn_annc_date_prev, c.earn_annc_date,
       COUNT(r.ret) AS n_days,
       EXP(SUM(LN(1 + r.ret))) - 1 AS cum_ret
FROM compustat AS c
LEFT JOIN crsp AS r
    ON r.permno = c.permno
   AND r.date > c.earn_annc_date_prev
   AND r.date <= c.earn_annc_date
GROUP BY c.name, c.fyear, c.permno, c.fiscal_year_end,
         c.earn_annc_date_prev, c.earn_annc_date
ORDER BY c.name, c.fyear
"""

duckdb.sql(query).df()

,name,fyear,permno,fiscal_year_end,earn_annc_date_prev,earn_annc_date,n_days,cum_ret
0,ADVANCED MICRO DEVICES,2000,61241,2000-12-31,2000-01-19,2001-01-17,225,0.107363
1,ADVANCED MICRO DEVICES,2001,61241,2001-12-31,2001-01-17,2002-01-16,239,-0.251275
2,ADVANCED MICRO DEVICES,2002,61241,2002-12-31,2002-01-16,2003-01-16,251,-0.617879
3,ADVANCED MICRO DEVICES,2003,61241,2003-12-31,2003-01-16,2004-01-20,253,1.413871
4,ADVANCED MICRO DEVICES,2004,61241,2004-12-31,2004-01-20,2005-01-18,251,-0.108166
...,...,...,...,...,...,...,...,...
589,WALMART INC,2021,55976,2022-01-31,2021-02-18,2022-02-17,253,0.024891
590,WALMART INC,2022,55976,2023-01-31,2022-02-17,2023-02-21,252,0.077497
591,WALMART INC,2023,55976,2024-01-31,2023-02-21,2024-02-20,250,0.211894
592,WALMART INC,2024,55976,2025-01-31,2024-02-20,2025-02-20,251,0.678052
